In [1]:
import pandas as pd 
import numpy as np
import os
import json
import re

In [2]:
"""
1. Read in the data
- Cazy

""" 

# 列出ProCAST.xlsx的所有sheet
print(pd.ExcelFile("ProCAST.xlsx").sheet_names)

['TCDB', 'KOG', 'DFVF', 'P450', 'CAZy', 'NR', 'SwissProt', 'TIGRFAM', 'SFLD', 'PHOBIUS', 'SIGNALP_EUK', 'SUPERFAMILY', 'PANTHER', 'GENE3D', 'HAMAP', 'PROSITE_PROFILES', 'PROSITE_PATTERNS', 'COILS', 'SMART', 'CDD', 'PRINTS', 'PFAM', 'MOBIDB_LITE', 'PIRSF', 'TMHMM', 'PRODOM', 'GO', 'PATHWAY', 'SeqLen']


In [3]:
# 读取ProCAST.xlsx
df = pd.read_excel("ProCAST.xlsx", sheet_name="CAZy")
df

,query_id,subject_id,identity,alignment length,mismatches,gap,q.start,q.end,s.start,s.end,e-value,bit score
0,XP_001728006.1,EAA29200.1 GH71: Glycoside Hydrolases (GHs) CB...,100.0,1046,0,0,1,1046,1,1046,0.000000e+00,2109.7
1,XP_001728007.1,EAA29200.1 GH71: Glycoside Hydrolases (GHs) CB...,98.4,318,5,0,1,318,1092,1409,2.500000e-168,595.1
2,XP_001728028.2,AEO57845.1 GT22: GlycosylTransferases (GTs),83.1,516,87,0,1,516,1,516,2.100000e-262,907.9
3,XP_001728031.1,EAA29319.1 GH93: Glycoside Hydrolases (GHs),100.0,453,0,0,1,453,1,453,2.300000e-265,917.5
4,XP_001728032.2,EAA29319.1 GH93: Glycoside Hydrolases (GHs),97.4,423,11,0,1,423,709,1131,1.200000e-229,798.9
...,...,...,...,...,...,...,...,...,...,...,...,...
797,XP_965749.1,XP_324789.1 GT32: GlycosylTransferases (GTs),100.0,372,0,0,1,372,1,372,5.100000e-202,706.8
798,XP_965769.3,QKX60254.1 GH76: Glycoside Hydrolases (GHs),60.9,780,292,8,46,818,7,780,6.200000e-280,966.8
799,XP_965782.3,XP_324822.1 GH35: Glycoside Hydrolases (GHs),97.8,1002,7,2,1,1001,1,988,0.000000e+00,1976.8
800,XP_965803.2,QDS75010.1 GT0: GlycosylTransferases (GTs),53.2,62,28,1,1,62,1,61,1.000000e-06,57.4


In [6]:
# 读取ProCAST.xlsx
df = pd.read_excel("ProCAST.xlsx", sheet_name="GO")
df.to_csv("ProCAST_GO.csv", index=False)

In [7]:
# 读取CSV文件
df = pd.read_csv('ProCAST_GO.csv')

# 拆分GO列中的多个条目
df['GO'] = df['GO'].str.split(';')

# 扩展DataFrame以包含多个条目
df = df.explode('GO')

# 拆分GO列中的信息
df[['TERM', 'ONTOLOGY', 'GO_id']] = df['GO'].str.split('|', expand=True)

# 选择所需的列
df = df[['id', 'GO_id', 'TERM', 'ONTOLOGY']]

df.rename(columns={'GO_id':'GO','ONTOLOGY':'GO_type', 'TERM':'ONTOLOGY'}, inplace=True)

df

,id,GO,ONTOLOGY,GO_type
0,XP_001727958.1,GO:0055114,oxidation-reduction process,BIOLOGICAL_PROCESS
0,XP_001727958.1,GO:0016702,"oxidoreductase activity, acting on single dono...",MOLECULAR_FUNCTION
1,XP_001727960.2,GO:0005515,protein binding,MOLECULAR_FUNCTION
2,XP_001727960.2,GO:0004190,aspartic-type endopeptidase activity,MOLECULAR_FUNCTION
2,XP_001727960.2,GO:0006508,proteolysis,BIOLOGICAL_PROCESS
...,...,...,...,...
8901,YP_009126729.1,GO:0004129,cytochrome-c oxidase activity,MOLECULAR_FUNCTION
8901,YP_009126729.1,GO:0016020,membrane,CELLULAR_COMPONENT
8902,YP_009126729.1,GO:0016021,integral component of membrane,CELLULAR_COMPONENT
8902,YP_009126729.1,GO:0016491,oxidoreductase activity,MOLECULAR_FUNCTION


In [8]:
go_df = df.groupby('id').agg({
    'GO': lambda x: ','.join(x),
    'ONTOLOGY': lambda x: ','.join(x),
    'GO_type': lambda x: ','.join(x)
}).reset_index()

go_df

,id,GO,ONTOLOGY,GO_type
0,XP_001727958.1,"GO:0055114,GO:0016702","oxidation-reduction process,oxidoreductase act...","BIOLOGICAL_PROCESS,MOLECULAR_FUNCTION"
1,XP_001727960.2,"GO:0005515,GO:0004190,GO:0006508","protein binding,aspartic-type endopeptidase ac...","MOLECULAR_FUNCTION,MOLECULAR_FUNCTION,BIOLOGIC..."
2,XP_001727961.1,GO:0005515,protein binding,MOLECULAR_FUNCTION
3,XP_001727970.2,GO:0005515,protein binding,MOLECULAR_FUNCTION
4,XP_001727974.1,"GO:0004190,GO:0006508","aspartic-type endopeptidase activity,proteolysis","MOLECULAR_FUNCTION,BIOLOGICAL_PROCESS"
...,...,...,...,...
5263,YP_009126725.1,"GO:0015986,GO:0015078,GO:0000276","ATP synthesis coupled proton transport,proton ...","BIOLOGICAL_PROCESS,MOLECULAR_FUNCTION,CELLULAR..."
5264,YP_009126726.1,"GO:0045263,GO:0015986,GO:0015078","proton-transporting ATP synthase complex, coup...","CELLULAR_COMPONENT,BIOLOGICAL_PROCESS,MOLECULA..."
5265,YP_009126727.1,"GO:0045263,GO:0015986,GO:0015078,GO:0004519","proton-transporting ATP synthase complex, coup...","CELLULAR_COMPONENT,BIOLOGICAL_PROCESS,MOLECULA..."
5266,YP_009126728.1,"GO:0045263,GO:0015986,GO:0015078,GO:0033177,GO...","proton-transporting ATP synthase complex, coup...","CELLULAR_COMPONENT,BIOLOGICAL_PROCESS,MOLECULA..."


In [10]:
# 将go_df的GO_type列中的MOLECULAR_FUNCTION改为MF
go_df['GO_type'] = go_df['GO_type'].str.replace('MOLECULAR_FUNCTION', 'MF')

# 将go_df的GO_type列中的BIOLOGICAL_PROCESS改为BP
go_df['GO_type'] = go_df['GO_type'].str.replace('BIOLOGICAL_PROCESS', 'BP')

# 将go_df的GO_type列中的CELLULAR_COMPONENT改为CC
go_df['GO_type'] = go_df['GO_type'].str.replace('CELLULAR_COMPONENT', 'CC')

go_df.to_csv('Go.csv', index=False)